# Pivoting

La noción del pivoting nace de darnos cuenta que para un problema binario es posible que cada restricción

$$\sum_{j\in\mathcal{N}}a_{ij}x_j\leq b_j$$ 

Puede ser reescrita como 

$$\sum_{j\in\mathcal{N}}a_{ij}x_j+s_i=b_i$$


La gracia del pivoting es intentar **forzar** que las variables $x_j$ del problema original terminen tomando los valores **exactamente 0 o 1**. Con esto, la idea es llegar a una base que sólo tenga variables de holgura $s$, implicando que todas las variables $x_j$ deberán estar en sus cotas. 

Así, intentaremos que las variables de holgura sean aquellas variables que siempre estén activas

Existen tres tipos de pivoteos:

- **Tipo 1: Pivote factible**: Entra una variable de holgura $s_i$ y sale una variable original $x_j$. Es el caso más idóneo que podríamos tener.

- **Tipo 2: Pivote de mejora de integralidad**: Entra y sale una variable del mismo tipo (por ejemplo, dos $x$ o dos $x$). Acá la factibilidad se mantiene. Elegimos el pivote tal que disminuya $I(x)$, donde $I(x)$ es la distancia hacia el vértice más cercano (0 o 1):

$$I(x)=\sum_{j\in\mathcal{N}}\min (x_j, 1-x_j)$$

- **Tipo 3: Pivote agresivo**: Permite romper la factibilidad primal de forma momentánea, haciendo que entre un $s_i$ con valor negativo. Esto es último recurso para poder sacar una variable $x_j$ de la base y seguir avanzando.

Cuando se hace un pivoteo de tipo 3 y se pierde la factibilidad, el algoritmo intenta realizar una complementación para intentar recuperar la factibilidad. En específico, definimos una medida de infactibilidad $P(x,s)$

$$P(x,s)=\sum_{j\in\mathcal{N}}\max(0,-x_j)+\sum_{i=1}^{n}\max (0,-s_i)$$

Por lo mismo, el procedimiento es:

1. Empieza complementando una variable no básica $x_j$
2. Si eso no mejora $P(x,s)$, prueba pares de variables.
3. Si no encuentra mejora, abandona la búsqueda.

Un ejemplo intuitivo nace con suponer el siguiente problema

$$x_1+x_2=1$$

Luego de varios pivoteos, se termina con

$$x_1 = 1.1\;\;\;\land\;\;\;x_2=0.3$$ 

La factibilidad se viola porque $1.1 + 0.3 = 1.4 > 1$. Si complementamos $x_1$ (cambiar $1.1$ a $0$), se llega a $x_1'=0,\;x_2'=0.3\implies 0.3<1$


Veamos como funciona cada tipo de pivoteo en un problema común y corriente. Primero, llamamos la instancia del knapsack

In [69]:

from math import isclose
import random
import helpers
import numpy as np

v, W, caps = helpers.instancia_knapsack(n=80, m=1, density=0.6, seed=42)
w = W[0]           # tomamos la única restricción
Wcap = caps[0]
n = len(v)

print("=== Instancia ===")
print("values :", np.round(v, 2))
print("weights:", w)
print("capacity:", Wcap)
print()


=== Instancia ===
values : [ 8.01 23.35 15.72 14.2  13.7  24.35  8.57 20.17 10.74  5.09 16.85 29.13
 14.1  19.62 17.77 18.84 13.81 12.46 19.06 17.99  7.72  9.95  7.9  27.73
 24.67 21.43 11.09 20.2  19.31 12.37  8.28  3.5   1.   18.35 24.62  7.19
 21.55 23.49 12.3  16.66  8.57 18.14 18.23  9.75  1.   27.91 11.17 24.05
 21.28 23.53 23.88  6.77  9.57 13.46  7.1   1.    9.88  2.29 22.13 15.29
 22.97 26.02  9.86 28.99  8.56  9.91 20.49  9.93  8.43  6.33 23.48  8.23
 11.23  1.   19.51 12.11 11.81  8.49 23.13 17.57]
weights: [ 8 35 31 22 22 39  8 32 13  8 26 44 34 35 33 36 25 10 38 23 25 19 12 42
 36 30 21 37 26 22 23 14  8 27 40  7 39 38 16 30 11 35 33 19  7 43 22 40
 32 36 35 12 19 23 24  6 26 11 34 32 41 34 19 43 21 18 41 19  8 23 36 12
 23 10 32 24 18 14 27 31]
capacity: 709



Nos definimos una función para resolver el lp relajado y obtener un punto inicial. Acá añadiremos **explícitamente** la variable slack.

In [70]:
def solve_lp_relaxation_gurobi(v, w, W):
    import gurobipy as gp
    from gurobipy import GRB
    m = gp.Model("knapLP")
    m.Params.OutputFlag = 0
    x = m.addVars(range(len(v)), lb=0, ub=1, name="x", vtype=GRB.CONTINUOUS)
    s = m.addVar(lb=0, name="s", vtype=GRB.CONTINUOUS)
    m.setObjective(gp.quicksum(v[i]*x[i] for i in range(len(v))), GRB.MAXIMIZE)
    m.addConstr(gp.quicksum(w[i]*x[i] for i in range(len(v))) + s == Wcap)
    m.optimize()
    return [x[i].X for i in range(len(v))], s.X
    
x, s = solve_lp_relaxation_gurobi(v, w, Wcap)


Y, algunas funciones helpers:

- `integral_infeasibility`: Calcula

  $$\sum(\min(x, 1-x))$$
- `capacity_violation`: Nos dice si se viola la capacidad. En específico, hacemos lo siguiente:

$$\sum w\cdot x - W_\text{cap}$$ 

Por lo mismo, si es negativo, no se viola la restricción. De lo contrario, hay violación.

- `is_integral`: Vemos si nuestras variables están cercanas al $1$ o al $0$.
- `clip`: Calcula

$$\max (0, \min (1, x))$$ 

In [18]:
def integral_infeasibility(x):
    return sum(min(xx, 1-xx) for xx in x)

def capacity_violation(x, w, Wcap):
    return sum(w[i]*x[i] for i in range(len(x))) - Wcap

def is_integral(x, tol=1e-6):
    return all(isclose(xx, 0.0, abs_tol=tol) or isclose(xx, 1.0, abs_tol=tol) for xx in x)

def clip(x):
    return max(0.0, min(1.0, x))

Ahora, nos definimos la función para hacer el pivoteo tipo 2. Así, mantendremos la factibilidad de capacidad y también disminuiremos la fraccionalidad de las soluciones.

1. Identificamos variables fraccionales $(0 < x_i < 1)$
2. Toma una de las más "centrales" (cerca de 0.5)
3. Elige otra variable $k$ para compensar el movimiento.
4. Mover $x_j$ hacia su extremo (0 o 1) y ajustar $x_k$ para mantener la igualdad.
5. Si con ese cambio, la medida de fraccionalidad $I(x)$ disminuye, se acepta.

In [49]:
def pivot_type2(x, w, Wcap):
    
    n = len(x)
    # Variables fraccionales: candidatas a pivotear
    frac = [i for i in range(n) if 1e-9 < x[i] < 1-1e-9]
    if len(frac) < 1:
        return False, x, "No hay fraccionales"

    # Priorizamos las más alejadas de los extremos (más “centradas”)
    frac.sort(key=lambda i: abs(x[i]-0.5), reverse=True)

    for j in frac:
        # Queremos empujar x_j hacia el 0 o el 1 (el extremo más cercano)
        target = 1.0 if x[j] > 0.5 else 0.0
        direction = 1 if target > x[j] else -1

        # Buscamos una segunda variable x_k para compensar el movimiento
        for k in range(n):
            if k == j or w[k] == 0:
                continue

            # Mantener sum(w_i x_i) = constante
            # implica mover x_k en proporción al peso w_j/w_k
            ratio = w[j]/w[k]*direction

            # Determinamos cuánto podemos mover (t máximo) antes de llegar a una cota
            # Para j: cuánto puede avanzar hasta 0 o 1
            t_j = (1-x[j]) if direction>0 else x[j]

            # Para k: límite antes de salirse de [0,1]
            if direction>0:
                t_k = x[k]*w[k]/w[j]
            else:
                t_k = (1-x[k])*w[k]/w[j]

            t = min(t_j, t_k)  # movimiento máximo posible

            if t <= 1e-12:
                continue

            # Aplica el movimiento tipo Simplex (sigue la arista factible)
            x_new = x[:]
            x_new[j] = clip(x[j] + direction*t)
            if direction>0:
                x_new[k] = clip(x[k] - t*(w[j]/w[k]))
            else:
                x_new[k] = clip(x[k] + t*(w[j]/w[k]))

            # Validamos que aún cumpla la capacidad
            if abs(capacity_violation(x_new, w, Wcap))>1e-6:
                continue

            # Si mejoró la fraccionalidad, lo aceptamos
            if integral_infeasibility(x_new) < integral_infeasibility(x):
                msg = f"[Tipo2] Empujo x[{j}]→{int(round(x_new[j]))} compensando con x[{k}]"
                return True, x_new, msg

    return False, x, "Sin pivote tipo 2 útil"

Y, ahora, la función para el pivote tipo 3. Sacrificamos factibiilidad momentáneamente y la reparamos. 

En el paso de complementing, se hace lo siguiente:
- Si tenemos sobrepeso (violación > 0), apagamos algunas variables (1->0) empezando por las de menor ratio valor/peso.
- Si tenemos holgura (violación < 0), prendemos variables (0->1) empezando por las de mayor ratio valor/peso. 

In [72]:
def pivot_type3_complementing(x, w, Wcap, v):
    
    # 1. Seleccionamos una variable fraccional representativa
    frac = [i for i in range(len(x)) if 1e-9 < x[i] < 1-1e-9]
    if not frac:
        return False, x, "Sin fraccionales"

    # Elegimos la más fraccional (más cerca de 0.5)
    j = max(frac, key=lambda i: min(x[i], 1-x[i]))
    target = 1.0 if x[j]>0.5 else 0.0

    # 2. Fijamos x_j al valor binario más cercano (0 o 1)
    x1 = x[:]
    x1[j] = target
    viol = capacity_violation(x1, w, Wcap)
    print(f"[Tipo3] Fijo x[{j}]→{int(target)}; violación={viol:+.3f}")

    # 3. Definimos los ratios v_i / w_i para priorizar ajustes
    ratio = [(i, v[i]/w[i]) for i in range(len(v))]

    # 4. Intentamos reparar factibilidad según el signo de la violación
    if viol>0:
        # Caso sobrepeso → debemos reducir la carga
        # Apagamos variables con peor rendimiento (menor v/w)
        for i,r in sorted(ratio, key=lambda z:z[1]):
            if x1[i]>0:
                delta = min(x1[i], viol/w[i])
                x1[i] = clip(x1[i]-delta)
                viol -= delta*w[i]
                print(f"  [Complementing] bajo x[{i}] {delta:.3f} (ratio={r:.2f}) → viol={viol:+.3f}")
                if viol<=1e-9: break
    elif viol<0:
        # Caso holgura → podemos llenar más la mochila
        # Encendemos variables con mejor rendimiento (mayor v/w)
        for i,r in sorted(ratio, key=lambda z:z[1], reverse=True):
            if x1[i]<1:
                delta = min(1-x1[i], -viol/w[i])
                x1[i] = clip(x1[i]+delta)
                viol += delta*w[i]
                print(f"  [Complementing] subo x[{i}] {delta:.3f} (ratio={r:.2f}) → viol={viol:+.3f}")
                if viol>=-1e-9: break

    # 5. Chequeamos si se logró restaurar factibilidad
    if abs(capacity_violation(x1,w,Wcap))<=1e-6:
        return True, x1, "[Tipo3+Complementing] Factible"
    return False, x, "[Tipo3+Complementing] Falló reparación"

Con esto, podemos hacer el bucle principal:

In [73]:
def obj(x,v): return sum(v[i]*x[i] for i in range(len(v)))

print("=== Iniciando heurística Pivoting ===")
print(f"I(x) inicial = {integral_infeasibility(x):.3f}, Obj={obj(x,v):.2f}\n")

for it in range(1,30):
    if is_integral(x) and abs(capacity_violation(x,w,Wcap))<=1e-9:
        print(f"[OK] Solución entera factible en {it-1} pasos.")
        break

    # ️Intentamos pivote tipo 2 (factible)
    moved,x_new,msg = pivot_type2(x,w,Wcap)
    if moved:
        print(f"Iter {it:02d} {msg} | I(x): {integral_infeasibility(x):.3f}->{integral_infeasibility(x_new):.3f}")
        x = x_new
        continue

    # ️Si no hay tipo 2 útil, probamos tipo 3 + complementing
    moved,x_new,msg = pivot_type3_complementing(x,w,Wcap,v)
    print(f"Iter {it:02d} {msg}")
    if moved:
        x = x_new
        continue

    # Si no hay mejora, redondeamos y terminamos
    print("Sin mejoras posibles → redondeo final.")
    x = [1.0 if xx>=0.5 else 0.0 for xx in x]
    break

print("\n=== Resultado final ===")
print("x* =", np.round(x,2))
print("Carga:", round(sum(w[i]*x[i] for i in range(len(x))),3), "/", Wcap)
print("Obj =", round(obj(x,v),3))
print("Fraccionalidad I(x) =", round(integral_infeasibility(x),4))

=== Iniciando heurística Pivoting ===
I(x) inicial = 0.073, Obj=508.19

[Tipo3] Fijo x[60]→0; violación=-3.000
  [Complementing] subo x[10] 0.115 (ratio=0.65) → viol=+0.000
Iter 01 [Tipo3+Complementing] Factible
Iter 02 [Tipo2] Empujo x[10]→0 compensando con x[2] | I(x): 0.115->0.097
Iter 03 [Tipo2] Empujo x[2]→0 compensando con x[5] | I(x): 0.097->0.077
Iter 04 [Tipo2] Empujo x[5]→0 compensando con x[34] | I(x): 0.077->0.075
Iter 05 [Tipo2] Empujo x[34]→0 compensando con x[60] | I(x): 0.075->0.073
[Tipo3] Fijo x[60]→0; violación=-3.000
  [Complementing] subo x[10] 0.115 (ratio=0.65) → viol=+0.000
Iter 06 [Tipo3+Complementing] Factible
Iter 07 [Tipo2] Empujo x[10]→0 compensando con x[2] | I(x): 0.115->0.097
Iter 08 [Tipo2] Empujo x[2]→0 compensando con x[5] | I(x): 0.097->0.077
Iter 09 [Tipo2] Empujo x[5]→0 compensando con x[34] | I(x): 0.077->0.075
Iter 10 [Tipo2] Empujo x[34]→0 compensando con x[60] | I(x): 0.075->0.073
[Tipo3] Fijo x[60]→0; violación=-3.000
  [Complementing] subo x[

Como se ve, se llega a una solución completamente entera. Sin embargo, es posible seguir mejorando el algoritmo. 

Ocurren veces que se llega a loops donde es difícil de salir. Por lo mismo, introducimos la variante con Tabú Search

In [58]:
from collections import deque

class TabuState:
    def __init__(self, tenure_pairs=8, tenure_fix=5):
        # pares (min(j,k), max(j,k)) usados en tipo 2
        self.tabu_pairs = deque(maxlen=tenure_pairs)
        # variables recién "fixeadas" en tipo 3 (índice j)
        self.tabu_fixed = deque(maxlen=tenure_fix)
        # mejores valores históricos para aspiración
        self.best_I = float('inf')
        self.best_obj = -float('inf')
        self.last_pair = None      # guarda (j,k) del último tipo 2 aceptado
        self.last_fixed = None     # guarda último j fijado por tipo 3


    def mark_pair(self, j, k):
        self.tabu_pairs.append((min(j,k), max(j,k)))

    def is_pair_tabu(self, j, k):
        return (min(j,k), max(j,k)) in self.tabu_pairs

    def mark_fixed(self, j):
        self.tabu_fixed.append(j)

    def is_fixed_tabu(self, j):
        return j in self.tabu_fixed

# Tenure pairs nos dirá por cuantas iteraciones los pares se mantendrán prohibidos
# Tenure fix son la lista de variables que no pueden volver a ser fijadas por otro pivote tipo 3

In [61]:
import random

def pivot_type2_tabu(x, w, Wcap, tabu, aspiration=True, eps_I=1e-3, shuffle_k=True):
    """
    Pivote tipo 2 con Tabú Search (pares (j,k)):
      - Mueve dos variables (j entra hacia 0/1; k compensa) para mantener sum(w_i x_i) = Wcap.
      - Requiere que la fraccionalidad I(x) mejore al menos eps_I (anti-microciclos).
      - Evita pares (j,k) recientemente usados (lista tabú).
      - Criterio de aspiración: permite un movimiento tabú si mejora el mejor I histórico (tabu.best_I).
    """
    n = len(x)

    # Variables fraccionales candidatas: 0 < x_j < 1
    frac = [j for j in range(n) if 1e-9 < x[j] < 1 - 1e-9]
    if not frac:
        return False, x, "Sin fraccionales"

    # Priorizamos las más “centrales” (las que más aportan a I(x))
    # Esto suele dar movimientos con mayor \delta I.
    frac.sort(key=lambda j: min(x[j], 1 - x[j]), reverse=True)

    for j in frac:
        # Empujar j hacia el extremo más cercano: 0 si x_j<0.5, 1 si x_j>0.5
        target = 1.0 if x[j] > 0.5 else 0.0
        direction = 1.0 if target > x[j] else -1.0  # +1 sube x_j, -1 baja x_j

        # 2) Construir la lista de candidatos k (compensador)
        ks = [k for k in range(n) if k != j and w[k] != 0]
        if shuffle_k:
            random.shuffle(ks)

        for k in ks:
            # --- Chequeo tabú de par (j,k) ---
            pair_is_tabu = tabu.is_pair_tabu(j, k)

            # Mantener sum(w_i x_i) constante ⇒
            #   x_j' = x_j + t*direction
            #   x_k' = x_k - t*direction*(w_j / w_k)
            # Limites:
            t_j = (1.0 - x[j]) if direction > 0 else x[j]
            if direction > 0:
                # x_k' = x_k - t*(w_j/w_k) >= 0 ⇒ t <= x_k * (w_k/w_j)
                t_k = x[k] * (w[k] / w[j])
            else:
                # x_k' = x_k + t*(w_j/w_k) <= 1 ⇒ t <= (1 - x_k) * (w_k/w_j)
                t_k = (1.0 - x[k]) * (w[k] / w[j])

            t = min(t_j, t_k)
            if t <= 1e-12:
                continue

            # Proponer movimiento
            x_new = x[:]
            x_new[j] = clip(x[j] + direction * t)
            if direction > 0:
                x_new[k] = clip(x[k] - t * (w[j] / w[k]))
            else:
                x_new[k] = clip(x[k] + t * (w[j] / w[k]))

            # Asegurar que seguimos con capacidad exacta (tolerancia numérica)
            if abs(capacity_violation(x_new, w, Wcap)) > 1e-7:
                continue

            I_old = integral_infeasibility(x)
            I_new = integral_infeasibility(x_new)
            dI = I_old - I_new

            # Aspiración: si el par es tabú, solo permitir si mejora histórico
            if pair_is_tabu:
                if aspiration and (I_new < tabu.best_I - 1e-12):
                    tabu.mark_pair(j, k)
                    return True, x_new, f"[Tipo2-TABU* asp] x[{j}]→{int(round(x_new[j]))} con x[{k}] (ΔI={dI:.4g})"
                else:
                    # Movimiento tabú sin aspiración: lo descartamos
                    continue

            # Aceptación normal: exigir mejora “real” de fraccionalidad
            if dI >= eps_I:
                tabu.mark_pair(j, k)  # registrar el par (anti-ciclo)
                return True, x_new, f"[Tipo2] x[{j}]→{int(round(x_new[j]))} con x[{k}] (ΔI={dI:.4g})"

    return False, x, "Sin pivote tipo 2 útil"

In [62]:
def pivot_type3_complementing_tabu(
    x, w, Wcap, v, tabu,
    spread=6,                 # repartir la reparación en más ítems ayuda a no caer al fallback
    avoid_j_in_fix=True,      # evita tocar j en la reparación
    forbid_alternation=True,  # evita descargar en la última fijada (corta ping-pong)
    allow_aspiration=False,   # por defecto NO aspiramos
    eps_I_asp=1e-4,
    verbose=True
):
    """
    Pivote tipo 3 + Complementing con Tabú.
    1) Elige una fraccional j NO tabú (cooldown). Si no hay, no hace tipo 3.
    2) Fija j al extremo más cercano (0/1), marca tabú de fijación.
       Solo si no alcanza, cae a un fallback que puede tocar los vetados.
    """
    n = len(x)
    tol = 1e-9

    # ============ Selección de j (fraccional) con cooldown duro ============
    frac = [i for i in range(n) if tol < x[i] < 1.0 - tol]
    if not frac:
        return False, x, "Sin fraccionales"

    # Excluir fraccionales en cooldown tabú
    pool = [i for i in frac if not tabu.is_fixed_tabu(i)]

    # Evitar alternancia simple: si existe última fijada y hay otras opciones, no elegirla
    if forbid_alternation and len(tabu.tabu_fixed) > 0:
        last_fixed_prev = tabu.tabu_fixed[-1]
        alt = [i for i in pool if i != last_fixed_prev]
        if alt:
            pool = alt

    # Si no queda ninguna candidata y NO hay aspiración → aborta tipo 3
    if not pool and not allow_aspiration:
        return False, x, "[Tipo3] Todas fraccionales en cooldown tabú"

    # Si se permite aspiración: acepta fraccional tabú solo si mejora I(x) al fijarla
    if not pool and allow_aspiration:
        # evaluar aspiración sobre todas las fraccionales
        best = None
        best_I = float("inf")
        for j_try in frac:
            target_try = 1.0 if x[j_try] >= 0.5 else 0.0
            x_probe = x[:]
            x_probe[j_try] = target_try
            I_probe = integral_infeasibility(x_probe)
            if I_probe < best_I - eps_I_asp:
                best_I = I_probe
                best = j_try
        if best is None:
            return False, x, "[Tipo3] Tabú total sin ganancia por aspiración"
        j = best
    else:
        # Elegimos la más fraccional (máximo min(x,1-x))
        j = max(pool, key=lambda i: min(x[i], 1.0 - x[i]))

    # ============ 2) Fijar j y marcar tabú ============
    target = 1.0 if x[j] >= 0.5 else 0.0
    x1 = x[:]
    x1[j] = target
    tabu.mark_fixed(j)          # actualiza tabu_fixed y guarda tabu.last_fixed = j

    # Violación respecto a la capacidad (positiva=sobrepeso)
    viol = sum(w[i]*x1[i] for i in range(n)) - Wcap
    if verbose:
        print(f"[Tipo3] Fijo x[{j}]→{int(target)}; violación={viol:+.3f}")

    # Si por casualidad no violó, acepta
    if abs(viol) <= tol:
        return True, x1, "[Tipo3] Sin violación; paso aceptado."

    # ============ 3) Complementing con 'ban' inteligente ============
    # Ratios v/w (asume w[i]>0 en items válidos)
    ratio = [(i, v[i]/w[i]) for i in range(n) if w[i] > 0]

    # Conjunto de veto para no deshacer el empuje ni el último patrón
    ban = {j}
    # Evitar descargar en la última fijada inmediatamente anterior (corta ping-pong j↔last_fixed_prev)
    if forbid_alternation and len(tabu.tabu_fixed) >= 2:
        last_fixed_prev = tabu.tabu_fixed[-2]   # la anterior a j
        ban.add(last_fixed_prev)
    # Evitar el 'otro' del último par de tipo 2 (si el último par involucraba a j)
    if tabu.last_pair is not None:
        a, b = tabu.last_pair
        other = b if j == a else (a if j == b else None)
        if other is not None:
            ban.add(other)

    # Listas núcleo vs fallback (fallback puede tocar vetados si no alcanza)
    ratio_core = [(i, r) for (i, r) in ratio if i not in ban]
    ratio_fallback = [(i, r) for (i, r) in ratio if i in ban]

    touched = 0

    if viol > 0:
        # --- Exceso de peso: bajar/apagar items de menor ratio primero ---
        for lst in (sorted(ratio_core, key=lambda z: z[1]),
                    sorted(ratio_fallback, key=lambda z: z[1])):

            for i, r in lst:
                if x1[i] <= 0.0:
                    continue
                delta = min(x1[i], viol / w[i])  # cuánto puedo bajar sin pasar 0, ni sobrecorregir
                if delta <= 0:
                    continue
                x1[i] = max(0.0, x1[i] - delta)
                viol -= delta * w[i]
                if verbose:
                    print(f"  [Complementing] bajo x[{i}] {delta:.3f} (ratio={r:.3f}) → viol={viol:+.3f}")
                if delta > 0:
                    touched += 1
                if abs(viol) <= tol or touched >= spread:
                    break
            if abs(viol) <= tol or touched >= spread:
                break

    else:  # viol < 0
        # --- Holgura: subir/encender items de mayor ratio primero ---
        for lst in (sorted(ratio_core, key=lambda z: z[1], reverse=True),
                    sorted(ratio_fallback, key=lambda z: z[1], reverse=True)):

            for i, r in lst:
                if x1[i] >= 1.0:
                    continue
                delta = min(1.0 - x1[i], (-viol) / w[i])  # cuánto puedo subir sin pasar 1, ni sobrecorregir
                if delta <= 0:
                    continue
                x1[i] = min(1.0, x1[i] + delta)
                viol += delta * w[i]
                if verbose:
                    print(f"  [Complementing] subo x[{i}] {delta:.3f} (ratio={r:.3f}) → viol={viol:+.3f}")
                if delta > 0:
                    touched += 1
                if abs(viol) <= tol or touched >= spread:
                    break
            if abs(viol) <= tol or touched >= spread:
                break

    # ============ Chequeo final de factibilidad ============
    if abs(sum(w[i]*x1[i] for i in range(n)) - Wcap) <= 1e-6:
        return True, x1, "[Tipo3+Complementing] Factible"
    else:
        return False, x, "[Tipo3+Complementing] Falló reparación"

In [74]:
tabu = TabuState(tenure_pairs=5, tenure_fix=3)
tabu.best_I = min(tabu.best_I, integral_infeasibility(x))
tabu.best_obj = max(tabu.best_obj, obj(x,v))
x, s = solve_lp_relaxation_gurobi(v, w, Wcap)


for it in range(1, 100):
    if is_integral(x) and abs(capacity_violation(x,w,Wcap))<=1e-9:
        print(f"[OK] Solución entera factible en {it-1} pasos.")
        break

    moved, x_new, msg = pivot_type2_tabu(x, w, Wcap, tabu, aspiration=True, eps_I=1e-3)
    if moved:
        x = x_new
        # actualizar mejores (aspiración)
        tabu.best_I = min(tabu.best_I, integral_infeasibility(x))
        tabu.best_obj = max(tabu.best_obj, obj(x,v))
        print(f"Iter {it:02d} {msg} | I(x)={integral_infeasibility(x):.4f}")
        continue

    moved, x_new, msg = pivot_type3_complementing_tabu(x, w, Wcap, v, tabu)
    print(f"Iter {it:02d} {msg}")
    if moved:
        x = x_new
        tabu.best_I = min(tabu.best_I, integral_infeasibility(x))
        tabu.best_obj = max(tabu.best_obj, obj(x,v))
        continue

    print("Sin mejoras → polishing.")
    x = [1.0 if xx>=0.5 else 0.0 for xx in x]
    break

print("\n=== Resultado final ===")
print("x* =", np.round(x,2))
print("Carga:", round(sum(w[i]*x[i] for i in range(len(x))),3), "/", Wcap)
print("Obj =", round(obj(x,v),3))
print("Fraccionalidad I(x) =", round(integral_infeasibility(x),4))

Iter 01 [Tipo2] x[10]→0 con x[39] (ΔI=0.01538) | I(x)=0.1000
Iter 02 [Tipo2] x[39]→0 con x[37] (ΔI=0.02105) | I(x)=0.0789
Iter 03 [Tipo2] x[37]→0 con x[47] (ΔI=0.003947) | I(x)=0.0750
Iter 04 [Tipo2] x[47]→0 con x[66] (ΔI=0.001829) | I(x)=0.0732
[Tipo3] Fijo x[66]→0; violación=-3.000
  [Complementing] subo x[10] 0.115 (ratio=0.648) → viol=+0.000
Iter 05 [Tipo3+Complementing] Factible
Iter 06 [Tipo2] x[10]→0 con x[59] (ΔI=0.02163) | I(x)=0.0938
Iter 07 [Tipo2] x[59]→0 con x[42] (ΔI=0.002841) | I(x)=0.0909
Iter 08 [Tipo2] x[42]→0 con x[66] (ΔI=0.01774) | I(x)=0.0732
Iter 09 [Tipo3] Todas fraccionales en cooldown tabú
Sin mejoras → polishing.

=== Resultado final ===
x* = [1. 1. 0. 0. 0. 0. 1. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 1. 1.
 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0.
 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0. 0. 1. 0. 1. 1.
 0. 0. 0. 0. 1. 0. 1. 0.]
Carga: 706.0 / 709
Obj = 506.506
Fraccionalidad I(x) = 0.0


Podemos ver que se llega a una solución mucho más rápida. 

A partir de la solución factible que se llega uno puede implementar algoritmos de búsqueda local.

# Line Search

La intuición de esta heurística se basa en **moverse a lo largo de una línea en el espacio de soluciones**, partiendo de un punto fraccional, buscando así una **solución entera factible**. 

Se toma un punto de referencia $\bar{x}$ y una dirección $\hat{x}-\bar{x}$ y se realiza una búsqueda a lo largo de la línea que une ambos puntos:

$$x(\epsilon_k)=\bar{x}+\epsilon_k(\hat{x}-\bar{x})\;\;\;\text{con}=0\epsilon_1<\dots<\epsilon_k=1$$ 

La idea es **evaluar distintos puntos** sobre esa línea para ver si alguno resulta:

- entero
- o factible

Por lo general, $\bar{x}$ es la solución de la relajación lineal del MIP, pero existen otras estrategias:

- **Hillier (1969)**: Desplazar los lados derechos (RHS / b) para hallar un punto **interior** al poliedro factible.
- **Ibaraki et al (1974)**: Buscar un punto interior, **maximizando holguras** y manteniendo fijo el valor objetivo.
- **Faalan y Hillier (1979)**_ Perturbar el vector $b$ **aleatoriamente**.

El algoritmo que veremos se llama **OCTANE**, que parte de una solución LP y define una dirección $d$ hacia un punto entero: 

$$x(\epsilon)=x^{LP}+\epsilon d$$ 

Sin embargo, a diferencia del line search clásico, OCTANE no elige arbitrariamente la dirección, sino que elige la dirección más cercana al vértice $[0,1]^n$ según una norma L1.

Así, se tiene el siguiente paso a paso: 

1. Partir del punto LP $x^{LP}$ y obtener su costo objetivo $c^Tx^{LP}$
2. Construir el octaedro centrado en $x^{LP}$ mediante la ecuación:
$$\sum^{n}_{j=1}\lvert x_j-x_{j}^{LP}\rvert=\delta$$
3. Identificar las direcciones hacia los vértices del cubo

$$
\sigma_j=\begin{cases}
+1&\text{si se desea mover}\;x_j\;\text{hacia}\;1\\ \\
-1&\text{si se desea mover}\;x_j\;\text{hacia}\;0
\end{cases}
$$

4. Buscar la dirección más prometedora:

$$d(\sigma)=\text{sign}(\sigma)\cdot (1-2x^{LP})$$

Selecciona la dirección que mejora el objetivo $c^Td$ y mantiene la factibilidad con $Ax\leq b$

5. Exploramos hacia esa dirección avanzando $x^{LP}+\epsilon d(\sigma)$ hasta alcanzar el límite del poliedro

6. El punto obtenido se proyecta al vértice binario más cercano


Veamos como se ve en el código


In [113]:
v, Wmat, caps = helpers.instancia_knapsack(n=50, m=1, density=0.6, seed=41)
w = Wmat[0]      # única restricción
Wcap = int(caps[0])
n = len(v)

print("=== Instancia (helpers.instancia_knapsack) ===")
print("values :", np.round(v, 2))
print("weights:", w)
print("capacity:", Wcap)
print()


=== Instancia (helpers.instancia_knapsack) ===
values : [22.22 22.88 21.77 30.12 18.94  6.14 11.04 20.13  9.   30.56 13.2   5.7
 26.7   9.04 15.72 17.41 11.25 10.42 21.8  12.51 27.03 20.77 11.2  27.99
 15.25  8.26  1.   10.64  7.7  14.25 11.55 18.24 18.75 26.39 22.49  6.24
 23.89 20.01 17.82 11.91 15.95 16.23 29.28 18.   16.52 20.02 15.85  6.9
  6.37 30.6 ]
weights: [31 43 38 35 33 10 15 38 18 38 12 18 44 25 38 29  7 18 25 29 44 34 18 39
 21 18  6 22 13 28 31 44 25 42 42  7 36 35 33 11 20 16 41 28 31 36 31 11
  9 42]
capacity: 475



Nos definimos la solución inicial

In [114]:
x_lp, s_lp = solve_lp_relaxation_gurobi(v, w, Wcap)
x_lp = np.array(x_lp, dtype=float)

print("x_lp =", np.round(x_lp, 3))
print("s_lp =", round(s_lp, 3))
print("Carga LP:", round(sum(w[i]*x_lp[i] for i in range(n)),3), "/", Wcap)
print()

x_lp = [1.    0.    0.    1.    0.    0.    1.    0.    0.    1.    1.    0.
 0.    0.    0.    0.    1.    0.    1.    0.    0.    0.    0.    1.
 1.    0.    0.    0.    0.    0.    0.    0.    1.    0.405 0.    1.
 1.    0.    0.    1.    1.    1.    1.    1.    0.    0.    0.    0.
 1.    1.   ]
s_lp = 0.0
Carga LP: 475.0 / 475



Funciones de utilidad

In [115]:
def obj(x, v):
    return float(np.dot(v, x))

def total_weight(x, w):
    return float(np.dot(w, x))

def capacity_violation(x, w, Wcap):
    return total_weight(x, w) - Wcap  # >0 sobrepeso

def integral_infeasibility(x):
    # I(x) = sum min(xi, 1-xi)
    return float(np.sum(np.minimum(x, 1.0 - x)))

def clip01(a):
    return np.clip(a, 0.0, 1.0)

Nos hacemos la función para construir $\hat{x}$

In [116]:
def build_hat_x(x_base, v, w, Wcap):
    """
    1) Redondea x_base a {0,1} por umbral 0.5.
    2) Repara si hay sobrepeso apagando ítems de peor ratio (v/w) hasta factibilizar.
    """
    xh = (x_base >= 0.5).astype(float)
    viol = capacity_violation(xh, w, Wcap)

    if viol > 1e-9:
        # Apaga por peor v/w primero (menor valor por unidad de peso)
        order = sorted(range(len(xh)), key=lambda i: (v[i]/w[i]) if w[i] > 0 else float("inf"))
        for i in order:
            if xh[i] == 1.0:
                xh[i] = 0.0
                viol -= w[i]
                if viol <= 1e-9:
                    break
    elif viol < -1e-9:
        # (Opcional) Si quieres llenar holgura prudente:
        order = sorted(range(len(xh)), key=lambda i: (v[i]/w[i]) if w[i] > 0 else -float("inf"), reverse=True)
        for i in order:
            if xh[i] == 0.0 and w[i] <= -viol:
                xh[i] = 1.0
                viol += w[i]
                if viol >= -1e-9:
                    break

    # Chequeo final de factibilidad (por si acaso)
    if capacity_violation(xh, w, Wcap) > 1e-6:
        # última salvaguarda: apaga lo que haga falta
        order = sorted(range(len(xh)), key=lambda i: (v[i]/w[i]) if w[i]>0 else float("inf"))
        for i in order:
            if xh[i] == 1.0:
                xh[i] = 0.0
                if capacity_violation(xh, w, Wcap) <= 1e-9:
                    break
    return xh

x_hat = build_hat_x(x_lp, v, w, Wcap)
print("=== Punto entero candidato (hat{x}) ===")
print("x_hat =", x_hat.astype(int))
print("Carga hat{x}:", total_weight(x_hat, w), "/", Wcap, "| Obj =", obj(x_hat, v))
print()


=== Punto entero candidato (hat{x}) ===
x_hat = [1 0 0 1 0 0 1 0 0 1 1 0 0 0 0 0 1 0 1 0 0 0 0 1 1 0 1 0 0 0 0 0 1 0 0 1 1
 0 0 1 1 1 1 1 0 0 0 1 1 1]
Carga hat{x}: 475.0 / 475 | Obj = 368.5451104946817



Y ahora ejecutamos el line search

In [117]:
def line_search(x_from, x_to, v, w, Wcap, grid=21, prefer_obj=True, print_grid=True):
    """
    Explora una grilla de ε en [0,1] y devuelve el mejor punto según:
      1) mínima fraccionalidad I(x)
      2) (desempate) mejor objetivo si prefer_obj=True
    Nota: si x_from y x_to son factibles (sum w x <= W), cualquier convex combo también lo será.
    """
    eps_vals = np.linspace(0.0, 1.0, grid)
    best = None
    rows = []

    for eps in eps_vals:
        x_eps = clip01((1.0 - eps) * x_from + eps * x_to)
        viol = capacity_violation(x_eps, w, Wcap)
        I = integral_infeasibility(x_eps)
        f = obj(x_eps, v)

        # Pequeña tolerancia por si hay redondeos numéricos
        feasible = (viol <= 1e-9)
        rows.append((eps, I, f, viol, feasible))

        # Criterio de selección: menor I; tie-break por mejor obj si prefer_obj
        if feasible:
            if best is None:
                best = (eps, x_eps, I, f)
            else:
                _, _, Ibest, fbest = best
                if I < Ibest - 1e-12 or (abs(I - Ibest) <= 1e-12 and prefer_obj and f > fbest + 1e-12):
                    best = (eps, x_eps, I, f)

    if print_grid:
        print("ε\tI(x)\t\tObj\t\tViol\tFactible")
        for eps, I, f, viol, feas in rows:
            print(f"{eps:0.2f}\t{I:0.6f}\t{f:0.3f}\t\t{viol:+0.3f}\t{'Si' if feas else '❌'}")
        print()

    if best is None:
        # Si no encontró factible (no debería ocurrir si ambos extremos lo son), vuelve al x_from
        return 0.0, x_from, integral_infeasibility(x_from), obj(x_from, v)
    return best  # (eps*, x*, I*, f*)

print("=== Line Search desde x_lp hacia x_hat ===")
eps_star, x_star, I_star, f_star = line_search(x_lp, x_hat, v, w, Wcap, grid=25)
print(f"Mejor ε* = {eps_star:0.3f} | I(x*) = {I_star:0.6f} | Obj = {f_star:0.3f}")
print("Carga x*:", round(total_weight(x_star, w),3), "/", Wcap)
print()

=== Line Search desde x_lp hacia x_hat ===
ε	I(x)		Obj		Viol	Factible
0.00	0.404762	371.323		+0.000	Si
0.04	0.471230	371.208		+0.000	Si
0.08	0.537698	371.092		+0.000	Si
0.12	0.604167	370.976		+0.000	Si
0.17	0.670635	370.860		+0.000	Si
0.21	0.737103	370.745		+0.000	Si
0.25	0.803571	370.629		+0.000	Si
0.29	0.870040	370.513		+0.000	Si
0.33	0.936508	370.397		+0.000	Si
0.38	1.002976	370.282		+0.000	Si
0.42	1.069444	370.166		+0.000	Si
0.46	1.135913	370.050		+0.000	Si
0.50	1.202381	369.934		+0.000	Si
0.54	1.102183	369.818		+0.000	Si
0.58	1.001984	369.703		+0.000	Si
0.62	0.901786	369.587		+0.000	Si
0.67	0.801587	369.471		+0.000	Si
0.71	0.701389	369.355		+0.000	Si
0.75	0.601190	369.240		+0.000	Si
0.79	0.500992	369.124		+0.000	Si
0.83	0.400794	369.008		+0.000	Si
0.88	0.300595	368.892		+0.000	Si
0.92	0.200397	368.777		+0.000	Si
0.96	0.100198	368.661		+0.000	Si
1.00	0.000000	368.545		+0.000	Si

Mejor ε* = 1.000 | I(x*) = 0.000000 | Obj = 368.545
Carga x*: 475.0 / 475



Nos quedamos con la solución que es directamente el redondeo $\hat{x}$. Interpretamos que el punto completamente entero $\epsilon=1$ llegó a ser el mejor según el criterio de fraccionalidad. 